# Spatial Bayesian Prompt Learning for Fine-Grained Aircraft Classification

This notebook demonstrates **Spatial-BPL**, an extension of Bayesian Prompt Learning
(Derakhshani et al., ICCV 2023) that improves fine-grained visual classification
by adding three complementary mechanisms:

1. **Bayesian Visual Adapter** — learns a distribution over image feature residuals
   (dual to BPL's text-side residuals)
2. **Patch-Level Spatial Attention** — discovers discriminative spatial regions via
   learned part queries cross-attending to ViT patch tokens
3. **Part-Aware Contrastive Loss** — encourages fine-grained discrimination at the
   part level

## Motivation

Standard BPL achieves strong generalization on many benchmarks by learning a
distribution over text prompt residuals. However, on FGVC-Aircraft (and similar
fine-grained datasets), performance remains limited because:

- **The visual side is frozen**: CLIP's ViT produces a single [CLS] token that
  captures global appearance. For aircraft, the differences are in *local spatial
  details* — engine types, window patterns, winglet shapes — that the [CLS] token
  may not emphasize.
- **Text-only adaptation is insufficient**: Changing the text prompt can steer
  CLIP toward different semantic concepts, but it cannot change *what visual
  features the image encoder extracts*.

Spatial-BPL addresses both limitations by adapting the visual representation
while preserving BPL's Bayesian text-side framework.

# 1. Setup

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install ftfy regex tqdm
# !pip install git+https://github.com/openai/CLIP.git

import os
import sys
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import clip
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('..'))

from spatial_bpl import SpatialBPL, SpatialBPLTrainer
from spatial_bpl.spatial_bpl_model import SpatialBPLForNewClasses
from spatial_bpl.trainer import split_base_new, make_fewshot_subset
from spatial_bpl.clip_patch_extractor import extract_patch_tokens

print(f"PyTorch version: {torch.__version__}")
print(f"CLIP available: {clip.__version__ if hasattr(clip, '__version__') else 'yes'}")

In [ ]:
# Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Reproducibility
SEED = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Load CLIP and FGVC-Aircraft Dataset

In [ ]:
# Load CLIP ViT-B/16
model_name = "ViT-B/16"
clip_model, preprocess = clip.load(model_name, device=device)

# Freeze CLIP
for p in clip_model.parameters():
    p.requires_grad_(False)

print(f"CLIP model: {model_name}")
print(f"Visual input resolution: {clip_model.visual.input_resolution}")
print(f"ViT embed dim: {clip_model.visual.conv1.out_channels}")
print(f"Output dim: {clip_model.visual.output_dim}")

In [ ]:
# Training transform with data augmentation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224, scale=(0.08, 1.0),
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

# Load FGVC-Aircraft
train_dataset = datasets.FGVCAircraft(
    root=os.path.expanduser("~/.cache"),
    download=True, split="train", transform=train_transform)

val_dataset = datasets.FGVCAircraft(
    root=os.path.expanduser("~/.cache"),
    download=True, split="val", transform=preprocess)

test_dataset = datasets.FGVCAircraft(
    root=os.path.expanduser("~/.cache"),
    download=True, split="test", transform=preprocess)

print(f"Train: {len(train_dataset)} images, {len(train_dataset.classes)} classes")
print(f"Val: {len(val_dataset)} images")
print(f"Test: {len(test_dataset)} images")

# 3. Dataset Preparation (Base/New Split)

In [ ]:
# Hyperparameters
SHOTS_PER_CLASS = 16
N_TOKENS = 4           # text prompt tokens
N_SAMPLES = 10         # MC samples
NUM_EPOCHS = 15
LR = 0.002
TRAIN_BATCH = 4        # Larger than BPL's 1 due to part contrastive loss
TEST_BATCH = 32
NUM_WORKERS = 4

# Spatial-BPL specific
ADAPTER_REDUCTION = 4  # Bottleneck reduction for visual adapter
ADAPTER_ALPHA = 0.2    # Residual blend ratio
NUM_PARTS = 6          # Number of learned part queries
KL_WEIGHT = 0.001      # KL divergence weight (same as BPL)
PART_WEIGHT = 0.1      # Part contrastive loss weight

print("Hyperparameters:")
for name, val in [
    ("shots_per_class", SHOTS_PER_CLASS),
    ("n_tokens", N_TOKENS),
    ("n_samples (MC)", N_SAMPLES),
    ("num_epochs", NUM_EPOCHS),
    ("lr", LR),
    ("train_batch", TRAIN_BATCH),
    ("adapter_reduction", ADAPTER_REDUCTION),
    ("adapter_alpha", ADAPTER_ALPHA),
    ("num_parts", NUM_PARTS),
    ("kl_weight", KL_WEIGHT),
    ("part_weight", PART_WEIGHT),
]:
    print(f"  {name}: {val}")

In [ ]:
# Create few-shot subset and base/new split
fewshot_train = make_fewshot_subset(train_dataset, SHOTS_PER_CLASS, seed=SEED)
print(f"Few-shot train size: {len(fewshot_train)}")

# Split into base (first half of classes) and new (second half)
m = len(fewshot_train) // 2
base_indices = fewshot_train.indices[:m]
new_indices = fewshot_train.indices[m:]

base_train = Subset(fewshot_train.dataset, base_indices)
base_val, new_val = split_base_new(val_dataset)
base_test, new_test = split_base_new(test_dataset)

base_classes = train_dataset.classes[:len(train_dataset.classes)//2]
new_classes = train_dataset.classes[len(train_dataset.classes)//2:]

print(f"Base classes: {len(base_classes)} | New classes: {len(new_classes)}")
print(f"Base train: {len(base_train)} | Base test: {len(base_test)}")
print(f"New test: {len(new_test)}")

In [ ]:
# DataLoaders
loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    drop_last=False,
    pin_memory=(device == "cuda"),
)

train_loader = DataLoader(
    base_train, batch_size=TRAIN_BATCH, shuffle=True, **loader_kwargs)
val_loader = DataLoader(
    base_val, batch_size=TEST_BATCH, shuffle=False, **loader_kwargs)
test_loader = DataLoader(
    base_test, batch_size=TEST_BATCH, shuffle=False, **loader_kwargs)
new_test_loader = DataLoader(
    new_test, batch_size=TEST_BATCH, shuffle=False, **loader_kwargs)

# 4. Initialize Spatial-BPL Model

In [ ]:
# Create the Spatial-BPL model
spatial_bpl = SpatialBPL(
    classnames=base_classes,
    clip_model=clip_model,
    n_tokens=N_TOKENS,
    n_samples=N_SAMPLES,
    device=device,
    adapter_reduction=ADAPTER_REDUCTION,
    adapter_alpha=ADAPTER_ALPHA,
    num_parts=NUM_PARTS,
    use_visual_adapter=True,
    use_spatial_attention=True,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in spatial_bpl.parameters() if p.requires_grad)
text_params = sum(p.numel() for name, p in spatial_bpl.named_parameters()
                  if p.requires_grad and ('ctx' in name or 'text_' in name))
visual_params = sum(p.numel() for name, p in spatial_bpl.named_parameters()
                    if p.requires_grad and 'visual_adapter' in name)
spatial_params = sum(p.numel() for name, p in spatial_bpl.named_parameters()
                     if p.requires_grad and 'spatial' in name)

print(f"\nTrainable parameters:")
print(f"  Text (BPL prompts):     {text_params:>8,}")
print(f"  Visual adapter:         {visual_params:>8,}")
print(f"  Spatial attention:       {spatial_params:>8,}")
print(f"  Total:                  {total_params:>8,}")
print(f"\nCLIP parameters (frozen): {sum(p.numel() for p in clip_model.parameters()):,}")

# 5. Train Spatial-BPL

In [ ]:
# Create trainer
trainer = SpatialBPLTrainer(
    model=spatial_bpl,
    clip_model=clip_model,
    device=device,
    lr=LR,
    num_epochs=NUM_EPOCHS,
    kl_weight=KL_WEIGHT,
    part_weight=PART_WEIGHT,
)

# Train
print("Starting Spatial-BPL training...")
print("=" * 70)
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    verbose=True,
)
print("=" * 70)
print("Training complete!")

# 6. Evaluate: Base and New Classes

In [ ]:
# Evaluate on base classes (seen during training)
base_acc = trainer.evaluate(test_loader, label_offset=0)
print(f"Base class accuracy (seen):   {base_acc:.4f}")

# Evaluate on new classes (unseen during training)
results = trainer.evaluate_base_to_new(
    new_classes=new_classes,
    new_test_loader=new_test_loader,
    base_test_loader=test_loader,
    label_offset=len(base_classes),
)

print(f"\nBase-to-New Generalization:")
print(f"  Base accuracy:  {results['base_acc']:.4f}")
print(f"  New accuracy:   {results['new_acc']:.4f}")
print(f"  Harmonic mean:  {2 * results['base_acc'] * results['new_acc'] / (results['base_acc'] + results['new_acc']):.4f}")

# 7. Comparison with Baselines

| Method | Base Acc | New Acc | HM |
|--------|---------|---------|----|
| Zero-shot CLIP | — | 23.13% | — |
| CoOp (4 tokens) | 39.67% | 25.25% | 30.84% |
| Unconditional BPL (4 tokens, 10 MC) | — | 34.35% | — |
| **Spatial-BPL (ours)** | **TBD** | **TBD** | **TBD** |

# 8. Visualize Part Attention Maps

One advantage of the spatial attention module is interpretability: we can
visualize which image regions each part query attends to. This shows the
model learning to focus on discriminative parts (engines, tails, windows).

In [ ]:
def visualize_part_attention(model, clip_model, images, class_names, labels,
                             num_images=4, device="cuda"):
    """Visualize what each part query attends to."""
    model.eval()

    # CLIP normalization stats for denormalization
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073])
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711])

    with torch.no_grad():
        _, patch_tokens = extract_patch_tokens(clip_model, images.to(device))
        maps = model.spatial_attention.get_part_attention_maps(
            patch_tokens.float()
        )  # [B, K, H, W]

    maps = maps.cpu()
    images = images.cpu()
    K = maps.shape[1]

    fig, axes = plt.subplots(
        num_images, K + 1, figsize=(3 * (K + 1), 3 * num_images)
    )
    if num_images == 1:
        axes = axes.unsqueeze(0)

    for i in range(min(num_images, images.shape[0])):
        # Denormalize image
        img = images[i].clone()
        for c in range(3):
            img[c] = img[c] * std[c] + mean[c]
        img = img.clamp(0, 1).permute(1, 2, 0).numpy()

        # Original image
        axes[i][0].imshow(img)
        axes[i][0].set_title(f"Class: {class_names[labels[i]]}")
        axes[i][0].axis('off')

        # Part attention maps
        for k in range(K):
            attn_map = maps[i, k].numpy()
            attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

            axes[i][k + 1].imshow(img)
            axes[i][k + 1].imshow(
                attn_map, alpha=0.6, cmap='hot',
                interpolation='bilinear',
                extent=[0, img.shape[1], img.shape[0], 0]
            )
            axes[i][k + 1].set_title(f"Part {k+1}")
            axes[i][k + 1].axis('off')

    plt.tight_layout()
    plt.show()

# Grab a batch and visualize
sample_images, sample_labels = next(iter(test_loader))
visualize_part_attention(
    spatial_bpl, clip_model, sample_images[:4],
    base_classes, sample_labels[:4], device=device
)

# 9. Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Total loss
axes[0].plot(history['train_loss'], label='Total Loss')
axes[0].plot(history['train_nll'], label='NLL', linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

# KL divergences
axes[1].plot(history['train_text_kl'], label='Text KL')
axes[1].plot(history['train_visual_kl'], label='Visual KL')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('KL Divergence')
axes[1].set_title('KL Divergence Components')
axes[1].legend()

# Part contrastive
axes[2].plot(history['train_part_loss'], label='Part Contrastive', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].set_title('Part-Aware Contrastive Loss')
axes[2].legend()

plt.tight_layout()
plt.show()

# 10. Ablation: Component Contributions

Let's compare different configurations to understand the contribution of
each component.

In [ ]:
def run_ablation(classnames, clip_model, train_loader, test_loader,
                 new_classes, new_test_loader, label_offset,
                 use_visual_adapter, use_spatial_attention,
                 device, config_name):
    """Run a single ablation configuration."""
    print(f"\n{'='*60}")
    print(f"Ablation: {config_name}")
    print(f"  visual_adapter={use_visual_adapter}, spatial_attention={use_spatial_attention}")
    print(f"{'='*60}")

    model = SpatialBPL(
        classnames=classnames,
        clip_model=clip_model,
        n_tokens=N_TOKENS,
        n_samples=N_SAMPLES,
        device=device,
        use_visual_adapter=use_visual_adapter,
        use_spatial_attention=use_spatial_attention,
    ).to(device)

    trainer = SpatialBPLTrainer(
        model=model, clip_model=clip_model, device=device,
        lr=LR, num_epochs=NUM_EPOCHS,
        kl_weight=KL_WEIGHT, part_weight=PART_WEIGHT,
    )

    trainer.train(train_loader, verbose=False)

    base_acc = trainer.evaluate(test_loader)
    results = trainer.evaluate_base_to_new(
        new_classes, new_test_loader, label_offset=label_offset
    )

    hm = 2 * base_acc * results['new_acc'] / (base_acc + results['new_acc'] + 1e-8)
    print(f"  Base: {base_acc:.4f} | New: {results['new_acc']:.4f} | HM: {hm:.4f}")
    return {'base': base_acc, 'new': results['new_acc'], 'hm': hm}

# Uncomment to run ablations (takes time)
# ablations = {}
#
# ablations['BPL only (baseline)'] = run_ablation(
#     base_classes, clip_model, train_loader, test_loader,
#     new_classes, new_test_loader, len(base_classes),
#     use_visual_adapter=False, use_spatial_attention=False,
#     device=device, config_name='BPL only (baseline)')
#
# ablations['BPL + Visual Adapter'] = run_ablation(
#     base_classes, clip_model, train_loader, test_loader,
#     new_classes, new_test_loader, len(base_classes),
#     use_visual_adapter=True, use_spatial_attention=False,
#     device=device, config_name='BPL + Visual Adapter')
#
# ablations['BPL + Spatial Attention'] = run_ablation(
#     base_classes, clip_model, train_loader, test_loader,
#     new_classes, new_test_loader, len(base_classes),
#     use_visual_adapter=False, use_spatial_attention=True,
#     device=device, config_name='BPL + Spatial Attention')
#
# ablations['Full Spatial-BPL'] = run_ablation(
#     base_classes, clip_model, train_loader, test_loader,
#     new_classes, new_test_loader, len(base_classes),
#     use_visual_adapter=True, use_spatial_attention=True,
#     device=device, config_name='Full Spatial-BPL')
#
# print("\n\n" + "="*60)
# print("ABLATION SUMMARY")
# print("="*60)
# for name, res in ablations.items():
#     print(f"{name:35s} | Base: {res['base']:.4f} | New: {res['new']:.4f} | HM: {res['hm']:.4f}")

print("Ablation code ready — uncomment cells above to run.")